In [ ]:
!pip install --upgrade setuptools pip

   ---------------------------------------- 0.0/818.2 kB ? eta -:--:--
   ---------------------------------------- 818.2/818.2 kB 11.2 MB/s  0:00:00
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 24.5 MB/s  0:00:00


ERROR: To modify pip, please run the following command:
C:\Users\DELL\Documents\Areas\MCD\Proyecto\estimador_estados\.venv\Scripts\python.exe -m pip install --upgrade setuptools pip


In [5]:
!pip install pypowsybl

   ---------------------------------------- 0.0/76.2 MB ? eta -:--:--
   - -------------------------------------- 3.1/76.2 MB 22.0 MB/s eta 0:00:04
   -- ------------------------------------- 4.2/76.2 MB 20.4 MB/s eta 0:00:04
   -- ------------------------------------- 4.2/76.2 MB 20.4 MB/s eta 0:00:04
   -- ------------------------------------- 5.0/76.2 MB 6.1 MB/s eta 0:00:12
   --- ------------------------------------ 7.6/76.2 MB 7.5 MB/s eta 0:00:10
   ---- ----------------------------------- 8.4/76.2 MB 8.2 MB/s eta 0:00:09
   ---- ----------------------------------- 9.4/76.2 MB 6.6 MB/s eta 0:00:11
   ------ --------------------------------- 12.6/76.2 MB 7.9 MB/s eta 0:00:09
   -------- ------------------------------- 15.7/76.2 MB 8.6 MB/s eta 0:00:08
   --------- ------------------------------ 17.6/76.2 MB 8.6 MB/s eta 0:00:07
   --------- ------------------------------ 18.9/76.2 MB 8.9 MB/s eta 0:00:07
   ----------- ---------------------------- 21.0/76.2 MB 8.7 MB/s eta 0:00:0

In [7]:
!pip freeze > ../requirements.txt

In [9]:
filepath = "../data/raw/rawx/IEEE14.rawx"

# JSON directo

In [10]:
import json
import pandas as pd

with open(filepath) as f:
    raw = json.load(f)

tables = {
    name: pd.DataFrame(block["data"], columns=block["fields"])
    for name, block in raw["network"].items()
    if isinstance(block.get("data"), list) and block["data"] and isinstance(block["data"][0], list)
}

buses = tables["bus"]
gens  = tables["generator"]
branches = tables["acline"]

In [3]:
buses

,ibus,name,baskv,ide,area,zone,owner,vm,va,nvhi,nvlo,evhi,evlo
0,1,BUS1,69.0,3,1,1,1,1.03000,0.0000,1.05,0.95,1.1,0.9
1,2,BUS2,69.0,2,2,1,1,1.00703,-5.4252,1.05,0.95,1.1,0.9
2,3,BUS3,69.0,2,1,1,1,0.98642,-14.0964,1.05,0.95,1.1,0.9
3,4,BUS4,69.0,1,1,1,1,0.98296,-11.4766,1.05,0.95,1.1,0.9
4,5,BUS5,69.0,1,1,1,1,0.98726,-9.8326,1.05,0.95,1.1,0.9
5,6,BUS6,138.0,2,2,2,2,0.99268,-16.9240,1.05,0.95,1.1,0.9
6,7,BUS7,138.0,1,2,2,2,0.99524,-15.3445,1.05,0.95,1.1,0.9
7,8,BUS8,69.0,2,2,2,2,1.03605,-15.3445,1.05,0.95,1.1,0.9
8,9,BUS9,138.0,1,2,2,2,0.97802,-17.3886,1.05,0.95,1.1,0.9
9,10,BUS10,138.0,1,2,2,2,0.97265,-17.6390,1.05,0.95,1.1,0.9


# Libreria PyPowSyBl

In [8]:
import pypowsybl as pp

'opf' extra dependencies are not installed, some features will not be available


In [ ]:
from pathlib import Path
RAWX_PATH = Path(filepath)

data = json.loads(RAWX_PATH.read_text())
data["general"]["version"] = "35.0"
data["network"]["caseid"]["data"][2] = 35
patched_path = Path("../data/interim/rawx/ieee14_rev35.rawx")
patched_path.write_text(json.dumps(data))
network = pp.network.load(str(patched_path))

In [22]:
buses = network.get_buses()
generators = network.get_generators()
loads = network.get_loads()
lines = network.get_lines()

buses

,name,v_mag,v_angle,connected_component,synchronous_component,voltage_level_id
id,,,,,,
VL1_0,,71.07000,0.0000,0,0,VL1
VL2_0,,69.48507,-5.4252,0,0,VL2
VL3_0,,68.06298,-14.0964,0,0,VL3
VL4_0,,67.82424,-11.4766,0,0,VL4
VL5_0,,68.12094,-9.8326,0,0,VL5
VL6_0,,136.98984,-16.9240,0,0,VL6
VL7_0,,137.34312,-15.3445,0,0,VL7
VL8_0,,71.48745,-15.3445,0,0,VL8
VL9_0,,134.96676,-17.3886,0,0,VL9


In [23]:
results = pp.loadflow.run_ac(network)
for r in results:
    print(r.connected_component_num, r.status, r.iteration_count)

network.get_buses()[["v_mag", "v_angle"]]

0 ComponentStatus.CONVERGED 5


,v_mag,v_angle
id,,
VL1_0,71.070000,0.000000
VL2_0,69.485239,-5.425504
VL3_0,68.063517,-14.095787
VL4_0,67.823887,-11.477071
VL5_0,68.120658,-9.833163
VL6_0,136.989706,-16.923067
VL7_0,137.343010,-15.345064
VL8_0,71.487063,-15.345064
VL9_0,134.966457,-17.389018
